In [2]:
%load_ext autoreload
%autoreload 2

from pipeline import EntityResolutionPipeline

er = EntityResolutionPipeline(sample_size=50000)
er.load_and_sample()

sample_s1 = er.sample_s1
sample_s2 = er.sample_s2
sample_s3 = er.sample_s3
sample_gt = er.sample_gt

Resolving dataset paths...
Loading datasets...
Creating 50000-entity sample...
Cleaning text strings...
Data loaded, sampled, and cleaned successfully!


In [4]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from tqdm.auto import tqdm

print("=== BUILDING PAIRWISE FEATURE MATRIX ===")

# 1. Create a balanced set of Positive (1) and Negative (0) pairs
np.random.seed(42)
pairs = []

# Create fast lookups
cand_pool = pd.concat([sample_s2, sample_s3]).set_index('entity_id')
cand_countries = cand_pool.groupby('country').groups

for _, row in tqdm(sample_gt.iterrows(), total=len(sample_gt), desc="Generating Pairs"):
    s1_id = row['source1_entity_id']
    s1_country = sample_s1.loc[sample_s1['entity_id'] == s1_id, 'country'].values[0]
    
    # Positive pairs (True Matches)
    raw_matches = row['matched_entity_ids']
    true_matches = [m.strip() for m in str(raw_matches).split(',') if m.strip()] if pd.notna(raw_matches) else []
    
    for match_id in true_matches:
        if match_id in cand_pool.index:
            pairs.append({'s1_id': s1_id, 'cand_id': match_id, 'label': 1})
            
    # Negative pairs (Hard Negatives - same country, not a match)
    # Sample up to 3 random candidates from the same country that are NOT true matches
    if s1_country in cand_countries:
        country_cands = cand_countries[s1_country]
        possible_negatives = list(set(country_cands) - set(true_matches))
        if possible_negatives:
            neg_samples = np.random.choice(possible_negatives, size=min(3, len(possible_negatives)), replace=False)
            for neg_id in neg_samples:
                pairs.append({'s1_id': s1_id, 'cand_id': neg_id, 'label': 0})

pairs_df = pd.DataFrame(pairs)
print(f"Generated {len(pairs_df):,} training pairs (Positives vs Negatives).")

# 2. Extract Features for the ML Model
def compute_features(row):
    s1_id = row['s1_id']
    cand_id = row['cand_id']
    
    # Get strings
    s1_name = str(sample_s1.loc[sample_s1['entity_id'] == s1_id, 'clean_name'].values[0])
    s1_addr = str(sample_s1.loc[sample_s1['entity_id'] == s1_id, 'clean_address'].values[0])
    
    cand_name = str(cand_pool.at[cand_id, 'clean_name'])
    cand_addr = str(cand_pool.at[cand_id, 'clean_address'])
    
    # Name Features
    name_exact = int(s1_name == cand_name and s1_name != "")
    name_ratio = fuzz.ratio(s1_name, cand_name)
    name_token_sort = fuzz.token_sort_ratio(s1_name, cand_name) # Handles word reordering ("Apple Inc" vs "Inc Apple")
    
    # Address Features
    addr_exact = int(s1_addr == cand_addr and s1_addr != "")
    addr_ratio = fuzz.ratio(s1_addr, cand_addr) if s1_addr and cand_addr else 0
    addr_token_set = fuzz.token_set_ratio(s1_addr, cand_addr) if s1_addr and cand_addr else 0 # Handles missing components
    
    return pd.Series([name_exact, name_ratio, name_token_sort, addr_exact, addr_ratio, addr_token_set])

print("Computing RapidFuzz string metrics... (This will take a minute)")
tqdm.pandas(desc="Extracting Features")
pairs_df[['name_exact', 'name_ratio', 'name_token_sort', 'addr_exact', 'addr_ratio', 'addr_token_set']] = pairs_df.progress_apply(compute_features, axis=1)

print("\nFeature Matrix Ready!")
display(pairs_df.head())

=== BUILDING PAIRWISE FEATURE MATRIX ===


Generating Pairs:   0%|          | 0/50000 [00:00<?, ?it/s]

Generated 322,784 training pairs (Positives vs Negatives).
Computing RapidFuzz string metrics... (This will take a minute)


Extracting Features:   0%|          | 0/322784 [00:00<?, ?it/s]


Feature Matrix Ready!


,s1_id,cand_id,label,name_exact,name_ratio,name_token_sort,addr_exact,addr_ratio,addr_token_set
0,S1-292935703,S2-195749344,1,0.0,52.631579,42.105263,0.0,92.307692,93.333333
1,S1-292935703,S2-617410789,1,0.0,80.000000,80.000000,0.0,81.632653,89.473684
2,S1-292935703,S2-516200703,1,0.0,68.965517,68.965517,0.0,81.632653,81.632653
3,S1-292935703,S3-421403180,1,1.0,100.000000,100.000000,0.0,70.175439,70.175439
4,S1-292935703,S3-942647343,1,0.0,77.777778,77.777778,0.0,81.355932,81.355932
